# TA-04: Peramalan Jumlah Kasus DBD Bulanan di Indonesia dengan LSTM

**Mata Praktikum:** Pembelajaran Mesin | **Modul 04:** LSTM untuk Data Time Series

**Dataset:** OpenDengue (https://opendengue.org/data.html) — kolom utama `calendar_start_date` (waktu) dan `dengue_total` (target).

**Alur:** Muat data → Bentuk deret bulanan nasional → EDA → Split kronologis → Scaling → Windowing → Model LSTM → Evaluasi (vs baseline) → Forecast ke depan → Simpan artefak untuk Flask.

> **Sebelum menjalankan:** unduh file CSV OpenDengue (disarankan *Highest temporal resolution data* atau *Spatial extract*), lalu letakkan di folder `data/` dan sesuaikan `DATA_PATH` di sel konfigurasi.

In [1]:
import os, json, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, Input
from keras.callbacks import EarlyStopping
warnings.filterwarnings("ignore")

# ===== KONFIGURASI =====
DATA_PATH   = "data/opendengue.csv"   # <-- ganti dengan nama file hasil unduhan Anda
COUNTRY     = "INDONESIA"
TIMESTEPS   = 12      # pakai 12 bulan terakhir untuk memprediksi bulan berikutnya
TRAIN_RATIO = 0.8
EPOCHS      = 200
BATCH_SIZE  = 16
SEED        = 42
ART_DIR     = Path("artifacts"); ART_DIR.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams["figure.figsize"] = (12, 5)

In [2]:
import sys
print(sys.executable)
import tensorflow as tf
print(tf.__file__)
import keras
print(keras.__version__)

c:\Users\windi nuralika repan\AppData\Local\Programs\Python\Python311\python.exe
c:\Users\windi nuralika repan\AppData\Local\Programs\Python\Python311\Lib\site-packages\tensorflow\__init__.py
3.15.1


## 1. Memuat Data & Diagnostik
OpenDengue memuat banyak resolusi spasial (`S_res`: Admin0/1/2) dan temporal (`T_res`: Year/Month/Week) dalam satu file. Kita periksa dulu apa yang tersedia untuk Indonesia sebelum memilih.

In [3]:
raw = pd.read_csv(DATA_PATH)
print("Kolom:", raw.columns.tolist())

df_id = raw[raw["adm_0_name"].str.upper() == COUNTRY].copy()
df_id["calendar_start_date"] = pd.to_datetime(df_id["calendar_start_date"])
df_id["calendar_end_date"]   = pd.to_datetime(df_id["calendar_end_date"])
print(f"\nBaris Indonesia: {len(df_id):,}")

# Ringkasan: kombinasi resolusi yang tersedia
summary = (df_id.groupby(["S_res", "T_res"])
                .agg(baris=("dengue_total", "size"),
                     mulai=("calendar_start_date", "min"),
                     akhir=("calendar_start_date", "max"))
                .reset_index())
summary

Kolom: ['adm_0_name', 'adm_1_name', 'adm_2_name', 'full_name', 'ISO_A0', 'FAO_GAUL_code', 'RNE_iso_code', 'IBGE_code', 'calendar_start_date', 'calendar_end_date', 'Year', 'dengue_total', 'case_definition_standardised', 'S_res', 'T_res', 'UUID', 'region']

Baris Indonesia: 6,230


,S_res,T_res,baris,mulai,akhir
0,Admin0,Month,6,2011-04-01,2024-10-01
1,Admin0,Year,2,2018-01-01,2022-01-01
2,Admin1,Month,5852,2010-01-01,2024-07-01
3,Admin1,Year,370,2010-01-01,2021-01-01


## 2. Membentuk Deret Waktu Bulanan Nasional
Strategi pemilihan (otomatis, berurutan):
1. `Admin0` + `Month` → langsung deret nasional bulanan.
2. `Admin1` + `Month` → jumlahkan seluruh provinsi per bulan (hanya bulan dengan cakupan provinsi lengkap).
3. Cadangan: `Week` → jumlahkan per bulan.

Jika hanya tersedia data **tahunan** untuk Indonesia, peramalan bulanan tidak mungkin dilakukan — gunakan file *Highest temporal resolution* / *Spatial extract*.

In [ ]:
def build_monthly_series(d: pd.DataFrame) -> (pd.Series, str):
    d = d.copy()
    d["month"] = d["calendar_start_date"].dt.to_period("M").dt.to_timestamp()

    # Opsi 1: nasional bulanan
    sel = d[(d["S_res"] == "Admin0") & (d["T_res"] == "Month")]
    if len(sel) >= 24:
        s = sel.groupby("month")["dengue_total"].sum()
        return s, "Admin0 + Month (nasional bulanan)"

    # Opsi 2: provinsi bulanan -> jumlahkan, buang bulan dengan cakupan provinsi tak lengkap
    sel = d[(d["S_res"] == "Admin1") & (d["T_res"] == "Month")]
    if len(sel) >= 24:
        g = sel.groupby("month").agg(total=("dengue_total", "sum"),
                                     n_prov=("adm_1_name", "nunique"))
        full = g["n_prov"].max()
        g = g[g["n_prov"] >= 33]
        print(f"Cakupan provinsi maksimum: {full}; bulan lolos filter cakupan >=90%: {len(g)}")
        return g["total"], "Admin1 + Month (dijumlahkan seluruh provinsi)"

    # Opsi 3: mingguan -> bulanan
    sel = d[d["T_res"] == "Week"]
    if len(sel) >= 100:
        lvl = "Admin0" if (sel["S_res"] == "Admin0").any() else sel["S_res"].iloc[0]
        sel = sel[sel["S_res"] == lvl]
        s = sel.groupby("month")["dengue_total"].sum()
        return s, f"{lvl} + Week -> dijumlahkan bulanan"

    raise ValueError("Data bulanan/mingguan Indonesia tidak ditemukan. Lihat tabel diagnostik di atas.")


series_raw, source_desc = build_monthly_series(df_id)
print("Sumber deret:", source_desc)

# Pastikan frekuensi bulanan teratur & cek bolong
full_idx = pd.date_range(series_raw.index.min(), series_raw.index.max(), freq="MS")
series = series_raw.reindex(full_idx)
series.index.name = "date"
print(f"Rentang: {series.index.min():%Y-%m} s.d. {series.index.max():%Y-%m} ({len(series)} bulan)")
print(f"Bulan hilang (NaN): {series.isna().sum()}")

Cakupan provinsi maksimum: 38; bulan lolos filter cakupan >=90%: 17
Sumber deret: Admin1 + Month (dijumlahkan seluruh provinsi)
Rentang: 2023-01 s.d. 2024-05 (17 bulan)
Bulan hilang (NaN): 0


In [5]:
# Penanganan bolong: interpolasi untuk celah pendek (<=3 bulan); jika masih ada NaN,
# ambil segmen kontinu terpanjang.
series = series.interpolate(method="linear", limit=3, limit_area="inside")

if series.isna().any():
    ok = series.notna().astype(int)
    grp = (ok.diff().fillna(0) != 0).cumsum()
    seg = ok.groupby(grp).agg(["sum", "first"])
    best = seg[seg["first"] == 1]["sum"].idxmax()
    series = series[grp == best]
    print("Masih ada celah panjang -> memakai segmen kontinu terpanjang.")

series = series.astype(float).clip(lower=0)
series.name = "cases"
print(f"Deret akhir: {series.index.min():%Y-%m} s.d. {series.index.max():%Y-%m}, n = {len(series)} bulan")
assert len(series) > TIMESTEPS * 3, "Data terlalu pendek untuk LSTM dengan TIMESTEPS ini."
series.head()

Deret akhir: 2023-01 s.d. 2024-05, n = 17 bulan


AssertionError: Data terlalu pendek untuk LSTM dengan TIMESTEPS ini.

In [6]:
s = series_raw.sort_index()
print("Jumlah bulan mentah:", len(s))
print(s.index.min(), "->", s.index.max())
print("Tahun yang ada:", sorted(s.index.year.unique()))
print(s.groupby(s.index.year).size())

Jumlah bulan mentah: 17
2023-01-01 00:00:00 -> 2024-05-01 00:00:00
Tahun yang ada: [2023, 2024]
month
2023    12
2024     5
Name: total, dtype: int64


In [7]:
d = df_id[(df_id["S_res"] == "Admin1") & (df_id["T_res"] == "Month")].copy()
d["month"] = d["calendar_start_date"].dt.to_period("M").dt.to_timestamp()
cov = d.groupby("month").agg(total=("dengue_total", "sum"), n_prov=("adm_1_name", "nunique"))
print(cov.groupby(cov.index.year).agg(
    bulan=("total", "size"), prov_min=("n_prov", "min"),
    prov_max=("n_prov", "max"), total_rata2=("total", "mean")).round(0))

       bulan  prov_min  prov_max  total_rata2
month                                        
2010      12        33        33      13007.0
2011      11        33        33       5535.0
2012      12        33        33       7520.0
2013      12        33        33       9376.0
2014      12        34        34       8362.0
2015      12        34        34      10807.0
2016      11        34        34      15641.0
2017      12        34        34       5697.0
2018      12        34        34       5466.0
2019      12        34        34      11510.0
2020      11        34        34       9596.0
2021      12        34        34       6126.0
2022      12        34        34      11939.0
2023      12        38        38       9560.0
2024       7        32        36      26668.0


## 3. EDA: Tren, Musiman, dan Pola Lain

In [ ]:
fig, ax = plt.subplots()
series.plot(ax=ax, label="Kasus bulanan")
series.rolling(12, center=True).mean().plot(ax=ax, color="red", lw=2, label="Rata-rata bergulir 12 bulan (tren)")
ax.set_title("Kasus DBD Bulanan di Indonesia (OpenDengue)")
ax.set_xlabel("Tanggal"); ax.set_ylabel("Jumlah kasus"); ax.grid(True); ax.legend()
plt.show()

In [ ]:
# Pola musiman: rata-rata per bulan kalender
seasonal = series.groupby(series.index.month).mean()
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(seasonal.index, seasonal.values, color="steelblue")
axes[0].set_xticks(range(1, 13)); axes[0].set_title("Rata-rata kasus per bulan kalender (musiman)")
axes[0].set_xlabel("Bulan"); axes[0].set_ylabel("Rata-rata kasus")

series.groupby(series.index.year).sum().plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Total kasus per tahun"); axes[1].set_xlabel("Tahun")
plt.tight_layout(); plt.show()

print(series.describe().round(1))
# Autokorelasi sederhana pada lag 1..24 untuk memperkuat justifikasi memori sekuens
acf = [series.autocorr(l) for l in range(1, 25)]
plt.figure(figsize=(10, 3.5)); plt.bar(range(1, 25), acf); plt.axhline(0, color="k", lw=.8)
plt.title("Autokorelasi (lag 1-24 bulan)"); plt.xlabel("Lag (bulan)"); plt.show()

## 4. Split Kronologis, Scaling, dan Windowing
Dua hal penting agar evaluasi jujur:
- **Split berdasarkan urutan waktu** (bukan acak).
- **Scaler hanya di-`fit` pada data latih** agar tidak ada kebocoran informasi (*data leakage*) dari data uji.

In [ ]:
values = series.values.reshape(-1, 1)
n_total = len(values)
split = int(n_total * TRAIN_RATIO)          # indeks pertama data uji

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(values[:split])                   # fit HANYA pada data latih
scaled = scaler.transform(values)

def make_sequences(arr, timesteps, start, end):
    # target berindeks start..end-1; input = timesteps nilai sebelum target
    X, y = [], []
    for i in range(start, end):
        X.append(arr[i - timesteps:i, 0])
        y.append(arr[i, 0])
    X, y = np.array(X), np.array(y)
    return X.reshape(X.shape[0], X.shape[1], 1), y      # (samples, timesteps, features)

X_train, y_train = make_sequences(scaled, TIMESTEPS, TIMESTEPS, split)
X_test,  y_test  = make_sequences(scaled, TIMESTEPS, split, n_total)   # konteks diambil dari ujung data latih
test_dates = series.index[split:]

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)
print(f"Latih: {series.index[0]:%Y-%m} s.d. {series.index[split-1]:%Y-%m}")
print(f"Uji  : {test_dates[0]:%Y-%m} s.d. {test_dates[-1]:%Y-%m}")

## 5. Membangun & Melatih Model LSTM

In [ ]:
model = Sequential([
    Input(shape=(TIMESTEPS, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1),
])
model.compile(optimizer="adam", loss="mean_squared_error")
model.summary()

In [ ]:
early = EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)

# validation_split mengambil BAGIAN AKHIR data latih (tetap kronologis); data uji tidak disentuh
history = model.fit(X_train, y_train,
                    epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_split=0.15, shuffle=False,
                    callbacks=[early], verbose=1)

plt.plot(history.history["loss"], label="loss (latih)")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title("Kurva Pembelajaran"); plt.xlabel("Epoch"); plt.ylabel("MSE (skala 0-1)")
plt.legend(); plt.grid(True); plt.show()

## 6. Evaluasi pada Data Uji
Kita bandingkan LSTM dengan dua **baseline sederhana**. Model deep learning baru bermakna jika mengalahkan baseline:
- **Naive:** prediksi bulan ini = nilai bulan lalu.
- **Seasonal naive:** prediksi bulan ini = nilai bulan yang sama tahun lalu.

In [ ]:
pred_scaled = model.predict(X_test, verbose=0)
y_pred   = scaler.inverse_transform(pred_scaled).flatten()          # kembali ke skala asli
y_actual = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

def metrics(y_true, y_hat):
    rmse = float(np.sqrt(mean_squared_error(y_true, y_hat)))
    mae  = float(mean_absolute_error(y_true, y_hat))
    nz = y_true > 0
    mape = float(np.mean(np.abs((y_true[nz] - y_hat[nz]) / y_true[nz])) * 100)
    return {"RMSE": rmse, "MAE": mae, "MAPE_%": mape}

# Baseline
all_vals = series.values
naive      = all_vals[split - 1:n_total - 1]                 # nilai bulan sebelumnya
seas_naive = all_vals[split - 12:n_total - 12]               # nilai bulan sama tahun lalu

results = pd.DataFrame({
    "LSTM":           metrics(y_actual, y_pred),
    "Naive (t-1)":    metrics(y_actual, naive),
    "Seasonal naive (t-12)": metrics(y_actual, seas_naive),
}).T.round(2)
results

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(test_dates, y_actual, "b-o", ms=4, label="Aktual")
plt.plot(test_dates, y_pred, "orange", marker="s", ms=4, label="Prediksi LSTM")
plt.plot(test_dates, naive, "g--", alpha=.6, label="Baseline naive")
plt.title("Kasus DBD Bulanan: Aktual vs Prediksi LSTM (Data Uji)")
plt.xlabel("Bulan"); plt.ylabel("Jumlah kasus"); plt.legend(); plt.grid(True); plt.show()

print(f"RMSE LSTM = {results.loc['LSTM','RMSE']:,.0f} kasus | MAE = {results.loc['LSTM','MAE']:,.0f} kasus")
print(f"Rata-rata kasus bulanan data uji = {y_actual.mean():,.0f} (untuk konteks besar error)")

## 7. Peramalan ke Depan (Recursive Multi-step)
Prediksi bulan ke-*t+1* dimasukkan kembali sebagai input untuk bulan ke-*t+2*, dst. **Galat menumpuk** semakin jauh horizon, jadi gunakan horizon pendek (≤ 6–12 bulan) dan tafsirkan sebagai estimasi tren, bukan angka pasti.

In [ ]:
def forecast_future(model, scaler, series, timesteps, n_months):
    window = scaler.transform(series.values[-timesteps:].reshape(-1, 1)).flatten().tolist()
    preds = []
    for _ in range(n_months):
        x = np.array(window[-timesteps:]).reshape(1, timesteps, 1)
        p = float(model.predict(x, verbose=0)[0, 0])
        preds.append(p); window.append(p)
    out = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten().clip(min=0)
    idx = pd.date_range(series.index[-1] + pd.offsets.MonthBegin(1), periods=n_months, freq="MS")
    return pd.Series(out, index=idx, name="forecast")

future = forecast_future(model, scaler, series, TIMESTEPS, 12)

plt.figure(figsize=(13, 5))
series.iloc[-48:].plot(label="Historis", color="steelblue")
future.plot(label="Forecast 12 bulan", color="red", marker="o")
plt.title("Peramalan Kasus DBD 12 Bulan ke Depan"); plt.legend(); plt.grid(True); plt.show()
future.round(0)

## 8. Menyimpan Artefak untuk Aplikasi Flask

In [ ]:
model.save(ART_DIR / "lstm_dbd.keras")
joblib.dump(scaler, ART_DIR / "scaler.joblib")
series.to_frame().to_csv(ART_DIR / "monthly_series.csv")

pd.DataFrame({"date": test_dates, "actual": y_actual, "predicted": y_pred}) \
  .to_csv(ART_DIR / "test_predictions.csv", index=False)

meta = {
    "timesteps": TIMESTEPS,
    "source": source_desc,
    "train_ratio": TRAIN_RATIO,
    "train_end": str(series.index[split - 1].date()),
    "test_start": str(test_dates[0].date()),
    "last_date": str(series.index[-1].date()),
    "metrics": {k: {m: round(v, 2) for m, v in row.items()} for k, row in results.to_dict("index").items()},
}
with open(ART_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(os.listdir(ART_DIR))

## 9. Kesimpulan (isi setelah melihat hasil Anda)
- Pola data: tren / musiman yang terlihat pada EDA (mis. puncak pada bulan-bulan musim hujan, lonjakan tahun tertentu).
- Kinerja: bandingkan RMSE/MAE LSTM terhadap baseline dan terhadap rata-rata kasus data uji.
- Keterbatasan: data nasional agregat, hanya satu fitur, adanya pelaporan tidak seragam antar-tahun/provinsi, dan galat menumpuk pada forecast multi-step.
- Pengembangan: tambah fitur iklim (curah hujan, suhu, kelembapan), coba GRU/Bidirectional LSTM, tuning `TIMESTEPS` dan unit.